# Day 1: Python Foundations and Pandas

This notebook introduces the Python building blocks used in data science, then applies them to the Titanic dataset. Run the cells from top to bottom in a fresh kernel.

## 1. Variables and basic data types

A variable gives a value a name. Python infers the value's type.

In [1]:
student_name = "Alice"
student_age = 18
average_score = 86.5
is_enrolled = True

print(student_name, student_age, average_score, is_enrolled)
print(type(student_name), type(student_age), type(average_score), type(is_enrolled))

Alice 18 86.5 True
<class 'str'> <class 'int'> <class 'float'> <class 'bool'>


Arithmetic operators compute values; comparison operators produce `True` or `False`.

In [2]:
completed_tasks = 3
remaining_tasks = 5 - completed_tasks
on_track = completed_tasks >= remaining_tasks
print(f"Remaining tasks: {remaining_tasks}; on track: {on_track}")

Remaining tasks: 2; on track: True


## 2. Lists and dictionaries

A list is an ordered collection. Indexing starts at zero, and negative indexes count from the end.

In [3]:
scores = [85, 90, 78, 92]
scores.append(88)
print("First:", scores[0])
print("Last:", scores[-1])
print("Average:", sum(scores) / len(scores))

First: 85
Last: 88
Average: 86.6


A dictionary stores key-value pairs and is useful for describing one record.

In [4]:
student = {"name": "Alice", "age": 18, "major": "Data Science"}
student["graduating"] = False
print(student["name"])
print(student)

Alice
{'name': 'Alice', 'age': 18, 'major': 'Data Science', 'graduating': False}


## 3. Conditions, loops, and functions

Conditions choose a branch based on a Boolean expression.

In [5]:
score = 85
if score >= 90:
    grade = "A"
elif score >= 80:
    grade = "B"
else:
    grade = "C or below"
print(grade)

B


A `for` loop repeats an operation for every item in a collection.

In [6]:
squared_scores = []
for value in scores:
    squared_scores.append(value ** 2)
print(squared_scores)

[7225, 8100, 6084, 8464, 7744]


A function packages reusable logic. The `return` statement sends a result back to the caller.

In [7]:
def letter_grade(score):
    if score >= 90:
        return "A"
    if score >= 80:
        return "B"
    if score >= 70:
        return "C"
    return "Needs improvement"

print([letter_grade(value) for value in scores])

['B', 'A', 'C', 'A', 'B']


### Check your understanding

The assertions below are small executable checks. If no error appears, the examples behave as expected.

In [8]:
assert scores[0] == 85
assert student["major"] == "Data Science"
assert letter_grade(92) == "A"
print("Python checks passed.")

Python checks passed.


## 4. Load the Titanic dataset with pandas

`pandas` represents tabular data with a `DataFrame`. This course uses the public seaborn-data CSV directly, so no seaborn package is required. All column names are lowercase. Loading this URL requires an Internet connection, and the remote source may change; the assertions below fail loudly if its expected 891-row schema changes.

In [9]:
import ssl
from pathlib import Path
from urllib.error import URLError

import pandas as pd

TITANIC_URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv"
fallback_candidates = [Path("../datasets/titanic.csv"), Path("datasets/titanic.csv")]
try:
    df = pd.read_csv(TITANIC_URL)
    data_source = TITANIC_URL
except (URLError, ssl.SSLError, OSError) as network_error:
    fallback_path = next((path for path in fallback_candidates if path.exists()), None)
    if fallback_path is None:
        raise FileNotFoundError("Titanic download failed and no local fallback was found.") from network_error
    df = pd.read_csv(fallback_path)
    data_source = str(fallback_path.resolve())

print(f"Data source: {data_source}")
df.columns = df.columns.str.lower()
expected_columns = [
    "survived", "pclass", "sex", "age", "sibsp", "parch", "fare",
    "embarked", "class", "who", "adult_male", "deck", "embark_town",
    "alive", "alone",
]
if df.shape != (891, 15):
    raise ValueError(f"Unexpected Titanic shape: {df.shape}")
if df.columns.tolist() != expected_columns:
    raise ValueError(f"Unexpected Titanic columns: {df.columns.tolist()}")
print(f"Rows: {df.shape[0]:,}; columns: {df.shape[1]}")
df.head()

Data source: https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv
Rows: 891; columns: 15


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


## 5. Inspect and select data

Start by checking column names, data types, and missing values.

In [10]:
print(df.columns.tolist())
summary = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
})
summary

['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'class', 'who', 'adult_male', 'deck', 'embark_town', 'alive', 'alone']


,dtype,missing
survived,int64,0
pclass,int64,0
sex,object,0
age,float64,177
sibsp,int64,0
parch,int64,0
fare,float64,0
embarked,object,2
class,object,0
who,object,0


In [11]:
df[["age", "fare", "survived"]].describe().round(2)

,age,fare,survived
count,714.00,891.00,891.00
mean,29.70,32.20,0.38
std,14.53,49.69,0.49
min,0.42,0.00,0.00
25%,20.12,7.91,0.00
50%,28.00,14.45,0.00
75%,38.00,31.00,1.00
max,80.00,512.33,1.00


Use brackets to select columns, `iloc` for row positions, and Boolean masks for conditions.

In [12]:
selected = df.loc[df["age"] > 30, ["sex", "age", "class", "fare"]]
selected.sort_values("fare", ascending=False).head()

,sex,age,class,fare
258,female,35.0,First,512.3292
679,male,36.0,First,512.3292
737,male,35.0,First,512.3292
438,male,64.0,First,263.0000
299,female,50.0,First,247.5208


In [13]:
rich_female_passengers = df.loc[(df["sex"] == "female") & (df["fare"] > 50)]
print("Matching rows:", len(rich_female_passengers))
rich_female_passengers.iloc[:3]

Matching rows: 87


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
31,1,1,female,NaN,1,0,146.5208,C,First,woman,False,B,Cherbourg,yes,False


## 6. Transform, clean, and aggregate

Use `.copy()` before transforming a subset or source table. This keeps the original `df` available for comparison.

In [14]:
clean_df = df.copy()
clean_df["family_size"] = clean_df["sibsp"] + clean_df["parch"] + 1
median_age = clean_df["age"].median()
clean_df["age"] = clean_df["age"].fillna(median_age)
clean_df = clean_df.drop(columns=["deck"], errors="ignore")
print("Missing ages after cleaning:", clean_df["age"].isna().sum())
clean_df[["sibsp", "parch", "family_size", "age"]].head()

Missing ages after cleaning: 0


,sibsp,parch,family_size,age
0,1,0,2,22.0
1,1,0,2,38.0
2,0,0,1,26.0
3,1,0,2,35.0
4,0,0,1,35.0


In [15]:
survival_by_sex = (
    clean_df.groupby("sex", observed=True)["survived"]
    .agg(passengers="size", survival_rate="mean")
    .assign(survival_rate=lambda table: table["survival_rate"].round(3))
)
survival_by_sex

,passengers,survival_rate
sex,,
female,314,0.742
male,577,0.189


In [16]:
class_summary = clean_df.groupby("class", observed=True).agg(
    passengers=("survived", "size"),
    average_age=("age", "mean"),
    average_fare=("fare", "mean"),
    survival_rate=("survived", "mean"),
).round(2)
class_summary

,passengers,average_age,average_fare,survival_rate
class,,,,
First,216,36.81,84.15,0.63
Second,184,29.77,20.66,0.47
Third,491,25.93,13.68,0.24


## 7. Export safely

`to_csv` can write a file. Here we generate CSV text in memory so running the lesson does not overwrite project data.

In [17]:
csv_preview = clean_df.head(3).to_csv(index=False)
print(csv_preview)

survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,embark_town,alive,alone,family_size
0,3,male,22.0,1,0,7.25,S,Third,man,True,Southampton,no,False,2
1,1,female,38.0,1,0,71.2833,C,First,woman,False,Cherbourg,yes,False,2
1,3,female,26.0,0,0,7.925,S,Third,woman,False,Southampton,yes,True,1



## Day 1 wrap-up

You used Python values, collections, control flow, and functions; then loaded, inspected, filtered, cleaned, grouped, and exported tabular data with pandas.